# 2. Additional: Quality control and denoising
This notebook compares a few different preprocessing methods.
### Import Packages

In [1]:
# Import all necessary packages
import IPython
import pandas as pd
import matplotlib.pyplot as plt
import os
import qiime2 as q2
from qiime2 import Visualization

%matplotlib inline

### Set Working Directory
Ensure that the working directory is correctly set to the 'scripts' folder within the main project directory. 
Otherwise, the file paths used in this notebook may not work properly.

In [2]:
# The working directory should normally default to the 'scripts' folder. 
# If it doesn't, set it manually using the command below.
# os.chdir("/home/jovyan/MicrobiomeAnalysis_TummyTribe/scripts")  # Adjust this path to match your folder structure.

# Verify that your working directory is the 'scripts' folder inside the main project directory (.../MicrobiomeAnalysis_TummyTribe/scripts)
cwd = os.getcwd()
if not cwd.endswith("MicrobiomeAnalysis_TummyTribe/scripts"):
    print("WARNING: The working directory is not set to the 'scripts' folder inside 'MicrobiomeAnalysis_TummyTribe'!")
    print("Current working directory:", cwd)
    print("Please set it manually using os.chdir().")
else:
    print(f"Working directory is correctly set to the 'scripts' folder (\"{cwd}\").")

Current working directory: /home/jovyan/MicrobiomeAnalysis_TummyTribe


In [5]:
# Data directories
data_in = "../data/raw"
data_out = "../data/additional/denoising"
results_dir = data_out

In [4]:
# Have a look at the sequencing data
! qiime tools peek $data_in/sequences-demux-paired.qza

UUID:        b4782ab7-550b-41f5-b906-ca2cda29ca9b
Type:        SampleData[PairedEndSequencesWithQuality]
Data format: SingleLanePerSamplePairedEndFastqDirFmt


## Quality Control

In [5]:
! qiime demux summarize \
    --i-data $data_in/sequences-demux-paired.qza \
    --o-visualization $results_dir/raw-QC.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: results/preprocessing/raw-QC.qzv


In [5]:
Visualization.load(f"{results_dir}/raw-QC.qzv")

<visualization: Visualization uuid: 8f69794d-57c4-4dba-89ed-c2fc5beed5e9>

## Denoising - Amplicon Sequence Variants

In [7]:
! qiime dada2 denoise-paired !help

Usage: qiime dada2 denoise-paired [OPTIONS]

  This method denoises paired-end sequences, dereplicates them, and filters
  chimeras.

Inputs:
  --i-demultiplexed-seqs ARTIFACT SampleData[PairedEndSequencesWithQuality]
                          The paired-end demultiplexed sequences to be
                          denoised.                                 [required]
Parameters:
  --p-trunc-len-f INTEGER Position at which forward read sequences should be
                          truncated due to decrease in quality. This truncates
                          the 3' end of the of the input sequences, which will
                          be the bases that were sequenced in the last cycles.
                          Reads that are shorter than this value will be
                          discarded. After this parameter is applied there
                          must still be at least a 12 nucleotide overlap
                          between the forward and reverse reads. If 0 is
            

In [8]:
! qiime dada2 denoise-paired \
    --i-demultiplexed-seqs $data_in/sequences-demux-paired.qza \
    --p-trunc-len-f 135 \
    --p-trunc-len-r 135 \
    --p-n-threads 3 \
    --o-table $data_out/dada2_table.qza \
    --o-representative-sequences $data_out/dada2_rep_seq.qza \
    --o-denoising-stats $data_out/dada2_stats.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureTable[Frequency] to: data/preprocessing/dada2_table.qza
Saved FeatureData[Sequence] to: data/preprocessing/dada2_rep_seq.qza
Saved SampleData[DADA2Stats] to: data/preprocessing/dada2_stats.qza


In [9]:
! qiime metadata tabulate \
    --m-input-file $data_out/dada2_stats.qza \
    --o-visualization $results_dir/dada2_stats.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: results/preprocessing/dada2_stats.qzv


In [6]:
Visualization.load(f"{results_dir}/dada2_stats.qzv")

<visualization: Visualization uuid: 378e8b20-dc98-49a7-a8b4-10da8dbf5d0a>

### Feature Table

In [11]:
! qiime feature-table summarize \
    --i-table $data_out/dada2_table.qza \
    --m-sample-metadata-file $data_in/metadata.tsv \
    --o-visualization $results_dir/dada2_table.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: results/preprocessing/dada2_table.qzv


In [7]:
Visualization.load(f"{results_dir}/dada2_table.qzv")

<visualization: Visualization uuid: 931f5e5c-287c-40c3-85ab-f788aa0e808a>

### With trunc 130

In [10]:
! qiime dada2 denoise-paired \
    --i-demultiplexed-seqs $data_in/sequences-demux-paired.qza \
    --p-trunc-len-f 130 \
    --p-trunc-len-r 130 \
    --p-n-threads 3 \
    --o-table $data_out/dada2_table_130.qza \
    --o-representative-sequences $data_out/dada2_rep_seq_130.qza \
    --o-denoising-stats $data_out/dada2_stats_130.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureTable[Frequency] to: data/preprocessing/dada2_table_130.qza
Saved FeatureData[Sequence] to: data/preprocessing/dada2_rep_seq_130.qza
Saved SampleData[DADA2Stats] to: data/preprocessing/dada2_stats_130.qza


In [ ]:
! qiime metadata tabulate \
    --m-input-file $data_out/dada2_stats_130.qza \
    --o-visualization $results_dir/dada2_stats_130.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: results/preprocessing/dada2_stats_130.qzv


In [12]:
Visualization.load(f"{results_dir}/dada2_stats_130.qzv")

<visualization: Visualization uuid: 5a3bc4eb-5824-458a-a8b8-d4008e39b1c8>

In [13]:
! qiime feature-table summarize \
    --i-table $data_out/dada2_table_130.qza \
    --m-sample-metadata-file $data_in/metadata.tsv \
    --o-visualization $results_dir/dada2_table_130.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: results/preprocessing/dada2_table_130.qzv


In [14]:
Visualization.load(f"{results_dir}/dada2_table_130.qzv")

<visualization: Visualization uuid: 62701662-f329-4330-90e6-fc15d105eb8b>

### with trunc 170/150

In [ ]:
! qiime dada2 denoise-paired \
    --i-demultiplexed-seqs $data_in/sequences-demux-paired.qza \
    --p-trunc-len-f 170 \
    --p-trunc-len-r 150 \
    --p-n-threads 3 \
    --o-table $data_out/dada2_table_170_150.qza \
    --o-representative-sequences $data_out/dada2_rep_seq_170_150.qza \
    --o-denoising-stats $data_out/dada2_stats_170_150.qza

In [ ]:
! qiime metadata tabulate \
    --m-input-file $data_out/dada2_stats_170_150.qza \
    --o-visualization $results_dir/dada2_stats_170_150.qzv

In [ ]:
Visualization.load(f"{results_dir}/dada2_stats_170_150.qzv")

In [ ]:
! qiime feature-table summarize \
    --i-table $data_out/dada2_table_170_150.qza \
    --m-sample-metadata-file $data_in/metadata.tsv \
    --o-visualization $results_dir/dada2_table_170_150.qzv

In [ ]:
Visualization.load(f"{results_dir}/dada2_table_170_150.qzv")

### with trunc 160/130

In [18]:
! qiime dada2 denoise-paired \
    --i-demultiplexed-seqs $data_in/sequences-demux-paired.qza \
    --p-trunc-len-f 160 \
    --p-trunc-len-r 130 \
    --p-n-threads 3 \
    --o-table $data_out/dada2_table_160_130.qza \
    --o-representative-sequences $data_out/dada2_rep_seq_160_130.qza \
    --o-denoising-stats $data_out/dada2_stats_160_130.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureTable[Frequency] to: data/preprocessing/dada2_table_160_130.qza
Saved FeatureData[Sequence] to: data/preprocessing/dada2_rep_seq_160_130.qza
Saved SampleData[DADA2Stats] to: data/preprocessing/dada2_stats_160_130.qza


In [19]:
! qiime metadata tabulate \
    --m-input-file $data_out/dada2_stats_160_130.qza \
    --o-visualization $results_dir/dada2_stats_160_130.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: results/preprocessing/dada2_stats_160_130.qzv


In [9]:
Visualization.load(f"{results_dir}/dada2_stats_160_130.qzv")

<visualization: Visualization uuid: cfdb9f50-8297-4d17-a012-b9a64906a9d3>

In [21]:
! qiime feature-table summarize \
    --i-table $data_out/dada2_table_160_130.qza \
    --m-sample-metadata-file $data_in/metadata.tsv \
    --o-visualization $results_dir/dada2_table_160_130.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: results/preprocessing/dada2_table_160_130.qzv


In [10]:
Visualization.load(f"{results_dir}/dada2_table_160_130.qzv")

<visualization: Visualization uuid: fde051e6-7ad4-4ad2-ae03-041bc7279195>

### with trunc 150

In [23]:
! qiime dada2 denoise-paired \
    --i-demultiplexed-seqs $data_in/sequences-demux-paired.qza \
    --p-trunc-len-f 150 \
    --p-trunc-len-r 150 \
    --p-n-threads 3 \
    --o-table $data_out/dada2_table_150.qza \
    --o-representative-sequences $data_out/dada2_rep_seq_150.qza \
    --o-denoising-stats $data_out/dada2_stats_150.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureTable[Frequency] to: data/preprocessing/dada2_table_135.qza
Saved FeatureData[Sequence] to: data/preprocessing/dada2_rep_seq_135.qza
Saved SampleData[DADA2Stats] to: data/preprocessing/dada2_stats_135.qza


In [24]:
! qiime metadata tabulate \
    --m-input-file $data_out/dada2_stats_150.qza \
    --o-visualization $results_dir/dada2_stats_150.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: results/preprocessing/dada2_stats_135.qzv


In [8]:
Visualization.load(f"{results_dir}/dada2_stats_150.qzv")

<visualization: Visualization uuid: d7fbfe66-db35-4f70-8625-7d2e2e2a8266>

In [26]:
! qiime feature-table summarize \
    --i-table $data_out/dada2_table_150.qza \
    --m-sample-metadata-file $data_in/metadata.tsv \
    --o-visualization $results_dir/dada2_table_150.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: results/preprocessing/dada2_table_135.qzv


In [9]:
Visualization.load(f"{results_dir}/dada2_table_150.qzv")

<visualization: Visualization uuid: a09ce8a3-d60b-4304-9ce4-ee0ad3f372b2>

### with trunc 140

! qiime dada2 denoise-paired \
    --i-demultiplexed-seqs $data_in/sequences-demux-paired.qza \
    --p-trunc-len-f 140 \
    --p-trunc-len-r 140 \
    --p-n-threads 3 \
    --o-table $data_out/dada2_table_140.qza \
    --o-representative-sequences $data_out/dada2_rep_seq_140.qza \
    --o-denoising-stats $data_out/dada2_stats_140.qza

In [ ]:
! qiime metadata tabulate \
    --m-input-file $data_out/dada2_stats_140.qza \
    --o-visualization $results_dir/dada2_stats_140.qzv

In [ ]:
Visualization.load(f"{results_dir}/dada2_stats_140.qzv")

In [ ]:
! qiime feature-table summarize \
    --i-table $data_out/dada2_table_140.qza \
    --m-sample-metadata-file $data_in/metadata.tsv \
    --o-visualization $results_dir/dada2_table_140.qzv

In [ ]:
Visualization.load(f"{results_dir}/dada2_stats_140.qzv")